# 02 — Chunking Demo

Full pipeline: **PDF → extract blocks → classify → chunk**

This notebook demonstrates Layers 1 → 2 → 4 of the HSC-Edu pipeline,
ending with semantic chunks ready for embedding.

In [1]:
from pathlib import Path

from hsc_edu.core.extraction import extract_document
from hsc_edu.core.classification import classify_blocks
from hsc_edu.core.chunking import chunk_blocks, count_tokens
from hsc_edu.core.models import BlockType

DATA_DIR = Path("../data")
SAMPLE_PDF = DATA_DIR / "Java.pdf"

print(f"PDF: {SAMPLE_PDF.name}  ({SAMPLE_PDF.stat().st_size / 1024 / 1024:.1f} MB)")

PDF: Java.pdf  (6.3 MB)


## 1. Extract blocks (Layer 1)

In [2]:
blocks = extract_document(SAMPLE_PDF, doc_id="java-demo")
print(f"Extracted {len(blocks)} blocks from '{SAMPLE_PDF.name}'")

Extracted 2121 blocks from 'Java.pdf'


## 2. Classify blocks (Layer 2)

In [3]:
classified = classify_blocks(blocks)

from collections import Counter

type_counts = Counter(b.block_type for b in classified)
print(f"Classified {len(classified)} blocks:")
for bt, cnt in type_counts.most_common():
    print(f"  {bt:<20s} {cnt:>5d}")

Classified 2121 blocks:
  paragraph             1897
  heading                210
  exercise                13
  note                     1


In [4]:
headings = [b for b in classified if b.block_type == BlockType.HEADING]
print(f"Headings found: {len(headings)}\n")

for h in headings[:15]:
    indent = "  " * ((h.heading_level or 1) - 1)
    text = h.raw_text.split("\n")[0][:80]
    print(f"  p.{h.page:>3d}  lv{h.heading_level}  {indent}{text}")

Headings found: 210

  p. 11  lv2    1.1. KHÁI NIỆM CƠ BẢN
  p. 12  lv2    1.2. ĐỐI TƯỢNG VÀ LỚP
  p. 14  lv2    1.3. CÁC NGUYÊN TẮC TRỤ CỘT
  p. 17  lv4        a) Quan hệ giữa một ngôi nhà và một bản thiết kế tương tự như quan hệ 
  p. 17  lv4        b) Khi mỗi đối tượng của một lớp giữ một bản riêng của một thuộc tính, 
  p. 19  lv2    2.1. ĐẶC TÍNH CỦA JAVA
  p. 20  lv3      2.1.1. Máy ảo Java – Java Virtual Machine
  p. 22  lv3      2.1.2. Các nền tảng Java
  p. 22  lv3      2.1.3. Môi trường lập trình Java
  p. 23  lv3      2.1.4. Cấu trúc mã nguồn Java
  p. 24  lv3      2.1.5. Chương trình Java đầu tiên
  p. 26  lv2    2.2. BIẾN
  p. 27  lv2    2.3. CÁC PHÉP TOÁN CƠ BẢN
  p. 27  lv3      2.3.1. Phép gán
  p. 27  lv3      2.3.2. Các phép toán số học


## 3. Chunk (Layer 4)

In [5]:
chunks = chunk_blocks(classified, subject="Lập trình Java")

print(f"Total chunks: {len(chunks)}")
print(f"Token range : {min(c.token_count for c in chunks)} – {max(c.token_count for c in chunks)}")
print(f"Avg tokens  : {sum(c.token_count for c in chunks) / len(chunks):.0f}")

subjects = {c.subject for c in chunks}
print(f"Subjects    : {subjects}")

Total chunks: 322
Token range : 12 – 1067
Avg tokens  : 549
Subjects    : {'Lập trình Java'}


### 3.1 Token distribution

In [6]:
buckets = {"0-64": 0, "64-256": 0, "256-512": 0, "512-1024": 0, ">1024": 0}
for c in chunks:
    t = c.token_count
    if t <= 64:
        buckets["0-64"] += 1
    elif t <= 256:
        buckets["64-256"] += 1
    elif t <= 512:
        buckets["256-512"] += 1
    elif t <= 1024:
        buckets["512-1024"] += 1
    else:
        buckets[">1024"] += 1

print("Token distribution:")
for label, cnt in buckets.items():
    bar = "█" * cnt
    print(f"  {label:>10s}: {cnt:>4d}  {bar}")

Token distribution:
        0-64:   87  ███████████████████████████████████████████████████████████████████████████████████████
      64-256:   20  ████████████████████
     256-512:   34  ██████████████████████████████████
    512-1024:  171  ███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
       >1024:   10  ██████████


## 4. First 10 chunks (with section path)

Each chunk shows its **heading hierarchy** (`section_path`) so you can see the table-of-contents context.

In [7]:
SHOW_N = 10

for i, ch in enumerate(chunks[:SHOW_N]):
    path_str = " > ".join(ch.section_path) if ch.section_path else "(no heading)"
    text_preview = ch.text[:200].replace("\n", " ↵ ")
    if len(ch.text) > 200:
        text_preview += " …"

    print(f"{'═' * 72}")
    print(f"Chunk {i}  │  {ch.token_count} tokens  │  pages {ch.page_start}–{ch.page_end}  │  type: {ch.block_type}")
    print(f"Subject: {ch.subject}")
    print(f"Chapter: {ch.chapter}")
    print(f"Section: {path_str}")
    print(f"Text   : {text_preview}")
    print()

════════════════════════════════════════════════════════════════════════
Chunk 0  │  1067 tokens  │  pages 0–1  │  type: paragraph
Subject: Lập trình Java
Chapter: 
Section: (no heading)
Text   : Mục lục ↵  ↵ GIỚI THIỆU .............................................................................5 ↵  ↵ Chương 1. MỞ ĐẦU ............................................................7 ↵  ↵ 1.1. KHÁI NIỆM CƠ BẢ …

════════════════════════════════════════════════════════════════════════
Chunk 1  │  1038 tokens  │  pages 1–2  │  type: paragraph
Subject: Lập trình Java
Chapter: 
Section: (no heading)
Text   : 5.6. BIẾN THỰC THỂ VÀ BIẾN ĐỊA PHƯƠNG ........... 80 ↵  ↵ Chương 6. SỬ DỤNG THƯ VIỆN JAVA ......................... 85 ↵  ↵ 6.1. ArrayList ..................................................................... …

════════════════════════════════════════════════════════════════════════
Chunk 2  │  1031 tokens  │  pages 2–3  │  type: paragraph
Subject: Lập trình Java
Chapter: 
Section: (no he

## 5. Inspect a specific chunk

Change `CHUNK_IDX` to drill into any chunk.

In [8]:
CHUNK_IDX = 150

ch = chunks[CHUNK_IDX]

print(f"chunk_id     : {ch.chunk_id}")
print(f"doc_id       : {ch.doc_id}")
print(f"subject      : {ch.subject}")
print(f"block_type   : {ch.block_type}")
print(f"token_count  : {ch.token_count}")
print(f"pages        : {ch.page_start} – {ch.page_end}")
print(f"chapter      : {ch.chapter}")
print(f"section      : {ch.section}")
print(f"section_path : {ch.section_path}")
print(f"block_ids    : {ch.block_ids}")
print(f"\n{'─' * 72}")
print(ch.text)

chunk_id     : f8947199a622
doc_id       : java-demo
subject      : Lập trình Java
block_type   : paragraph
token_count  : 1006
pages        : 116 – 118
chapter      : 7.11. CÁC MỨC TRUY NHẬP
section      : 7.11. CÁC MỨC TRUY NHẬP
section_path : ['7.11. CÁC MỨC TRUY NHẬP']
block_ids    : ['86cd7478d187', '29d90afac488', 'a383048133d7', '138f11926e22', '391d31194e4c', '25d4bcd9f7b4', 'f046145d5fd1', '9faa30ae2084', '02e12f375a48', 'e63089b03d57', 'd972f47b750d']

────────────────────────────────────────────────────────────────────────
7.11. CÁC MỨC TRUY NHẬP

public và private là hai mức được sử dụng nhiều nhất. Mức public thường dùng 
cho các lớp, hằng (biến static final, xem chi tiết tại Mục 10.6), các phương thức dành 
cho mục đích tương tác với bên ngoài (ví dụ các phương thức get và set), và hầu hết 
các hàm khởi tạo. private được dùng cho hầu hết các biến thực thể và cho các 
phương thức mà ta không muốn được gọi từ bên ngoài lớp (các phương thức dành 
riêng cho các phương thức pu